# K-Means Clustering on an Unsupervised Dataset

This notebook reads an unlabeled dataset and groups its rows by similarity using **k-means clustering**.

**Workflow**
1. Load the data (your CSV, or a built-in sample if none is provided)
2. Explore and clean it
3. Scale the features
4. Choose *k* using the Elbow method and Silhouette score
5. Fit k-means and assign cluster labels
6. Visualise (PCA 2-D projection) and profile each cluster
7. Save the clustered data to CSV

## 1. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score, calinski_harabasz_score

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 2. Load the dataset

Set `DATA_PATH` to your CSV file (e.g. `"customers.csv"`).  
If the path is `None` or the file doesn't exist, a sample dataset (Iris measurements, **labels discarded**) is used so the notebook runs end-to-end.

In [ ]:
import os
from sklearn.datasets import load_iris

DATA_PATH = None   # <-- e.g. "my_data.csv"

if DATA_PATH and os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    print(f"Loaded {DATA_PATH}")
else:
    iris = load_iris(as_frame=True)
    df = iris.frame.drop(columns="target")      # drop labels -> unsupervised
    df.columns = [c.replace(" (cm)", "") for c in df.columns]
    print("Using sample dataset (Iris, labels removed)")

print("Shape:", df.shape)
df.head()

## 3. Explore and clean

In [ ]:
df.info()
print("\nMissing values per column:\n", df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
df.describe().T

In [ ]:
# Keep numeric columns only (k-means uses Euclidean distance), drop duplicates,
# and fill missing values with the column median.
X_df = df.select_dtypes(include=[np.number]).copy()

# Optionally drop ID-like columns here, e.g.: X_df = X_df.drop(columns=["CustomerID"])
X_df = X_df.drop_duplicates()
X_df = X_df.fillna(X_df.median())

print("Features used for clustering:", list(X_df.columns))
print("Rows:", len(X_df))

In [ ]:
# Feature distributions and correlations
X_df.hist(figsize=(10, 6), bins=20, edgecolor="black")
plt.suptitle("Feature distributions"); plt.tight_layout(); plt.show()

plt.figure(figsize=(6, 4.5))
sns.heatmap(X_df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation matrix"); plt.show()

## 4. Scale the features

K-means is distance-based, so features on larger scales would dominate. Standardising gives each feature zero mean and unit variance.

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(X_df)
X[:3]

## 5. Choose the number of clusters (k)

- **Elbow method** – look for the point where inertia stops dropping sharply.
- **Silhouette score** – higher is better (range −1 to 1).
- **Davies–Bouldin** – lower is better. **Calinski–Harabasz** – higher is better.

In [ ]:
k_range = range(2, 11)
rows = []
for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X)
    rows.append({
        "k": k,
        "inertia": km.inertia_,
        "silhouette": silhouette_score(X, km.labels_),
        "davies_bouldin": davies_bouldin_score(X, km.labels_),
        "calinski_harabasz": calinski_harabasz_score(X, km.labels_),
    })
metrics = pd.DataFrame(rows).set_index("k")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(metrics.index, metrics["inertia"], "o-")
ax[0].set(title="Elbow method", xlabel="k", ylabel="Inertia (WCSS)")
ax[1].plot(metrics.index, metrics["silhouette"], "o-", color="darkorange")
ax[1].set(title="Silhouette score", xlabel="k", ylabel="Score")
plt.tight_layout(); plt.show()

metrics.round(3)

In [ ]:
# Pick k: best silhouette by default. Override manually after inspecting the elbow plot if you prefer.
best_k = int(metrics["silhouette"].idxmax())
K = best_k
print(f"Selected k = {K}")

## 6. Fit the final k-means model

In [ ]:
kmeans = KMeans(n_clusters=K, n_init=20, random_state=RANDOM_STATE)
labels = kmeans.fit_predict(X)

result = X_df.copy()
result["cluster"] = labels

print("Silhouette score:", round(silhouette_score(X, labels), 3))
print("\nCluster sizes:")
print(result["cluster"].value_counts().sort_index())

## 7. Visualise the clusters

In [ ]:
# Project to 2-D with PCA for plotting
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_2d = pca.fit_transform(X)
centers_2d = pca.transform(kmeans.cluster_centers_)

plt.figure(figsize=(7, 5.5))
sns.scatterplot(x=X_2d[:, 0], y=X_2d[:, 1], hue=labels, palette="tab10", s=50, edgecolor="white")
plt.scatter(centers_2d[:, 0], centers_2d[:, 1], c="black", marker="X", s=250, label="Centroids")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%} variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%} variance)")
plt.title(f"K-Means clusters (k={K}) in PCA space")
plt.legend(title="Cluster"); plt.show()

In [ ]:
# Silhouette plot: how well each point fits its cluster
sil_vals = silhouette_samples(X, labels)
fig, ax = plt.subplots(figsize=(7, 5))
y_lower = 10
for c in range(K):
    v = np.sort(sil_vals[labels == c])
    y_upper = y_lower + len(v)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, v, alpha=0.8)
    ax.text(-0.05, y_lower + 0.5 * len(v), str(c))
    y_lower = y_upper + 10
ax.axvline(silhouette_score(X, labels), color="red", linestyle="--", label="Mean")
ax.set(title="Silhouette plot", xlabel="Silhouette coefficient", ylabel="Cluster")
ax.set_yticks([]); ax.legend(); plt.show()

In [ ]:
# Pairwise feature view coloured by cluster (first up to 4 features)
cols = list(X_df.columns[:4]) + ["cluster"]
sns.pairplot(result[cols], hue="cluster", palette="tab10", corner=True, plot_kws={"s": 25})
plt.show()

## 8. Profile the clusters

In [ ]:
# Mean of each feature per cluster (original units)
profile = result.groupby("cluster").mean().round(2)
profile["size"] = result["cluster"].value_counts().sort_index()
profile

In [ ]:
# Standardised centroids heatmap: which features define each cluster?
centroids = pd.DataFrame(kmeans.cluster_centers_, columns=X_df.columns)
plt.figure(figsize=(1.5 * len(X_df.columns) + 2, 0.8 * K + 2))
sns.heatmap(centroids, annot=True, cmap="RdBu_r", center=0, fmt=".2f")
plt.title("Cluster centroids (standardised features)")
plt.ylabel("Cluster"); plt.show()

## 9. Save the results

In [ ]:
OUTPUT_PATH = "clustered_data.csv"
result.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(result)} rows to {OUTPUT_PATH}")

## Notes

- **Different data:** set `DATA_PATH`, and drop ID columns in section 3. Encode categorical columns (e.g. `pd.get_dummies`) if you want them included.
- **Choosing k:** the code defaults to the best silhouette score; set `K` manually if domain knowledge suggests otherwise.
- **Limitations:** k-means assumes roughly spherical, similarly sized clusters and is sensitive to outliers. If results look poor, consider DBSCAN or Gaussian mixture models.